In [ ]:
import pandas as pd
from sqlalchemy import select

from app.config import settings
from app.db import get_session
from app.models import Attributes, NearbyParcelIdsParams
from app.nearby_parcel_request import fetch_nearby_ids

params = NearbyParcelIdsParams(
    lon=-79.9513103495131,
    lat=32.706641645949055,
    distance=5.0,
)

nearby = fetch_nearby_ids(params)

len(nearby.objectIds), nearby.objectIds[:10]

In [7]:
session = next(get_session())

stmt = select(Attributes).where(
    Attributes.objectid.in_(nearby.objectIds)
)

rows = session.scalars(stmt).all()

In [8]:
df = pd.DataFrame(
    [
        {
            column.name: getattr(row, column.name)
            for column in Attributes.__table__.columns
        }
        for row in rows
    ]
)

df.head()

,id,objectid,pid,owner1,owner2,tax_district,class_code,mail_st_no,mail_st_name,mail_st_type,...,subdivision,acreage,legal_residence,other,agr,deed_book_page,plat_book_page,sale_price,recorded_date,doc_date
0,7,7,4260400066,WYNNE CLAYTON ASHWORTH,NaN,3-4,101 - RESID-SFR,131,OYSTER POINT ROW,NaN,...,NaN,0.00,Y,N,N,0173-698,AX-53,152500.0,2011-02-24,2011-02-22
1,10,10,4310200094,SYRETT JUDSON,SYRETT SAMANTHA,3-4,101 - RESID-SFR,1326,FORT LAMAR,RD,...,NaN,0.00,Y,N,N,0442-567,J-30,319000.0,2014-11-24,2014-11-10
2,14,14,4541300050,HANVEY WILLIAM H,HANVEY JEAN H,3-8,101 - RESID-SFR,783,LAROCHE,ST,...,NaN,0.00,Y,N,N,D319-128,L-87,1.0,1999-01-28,1999-01-25
3,22,22,3251500087,CULLER JAMES,CULLER JENNIFER,3-3,750 - SPCLTY-REC,10,NaN,NaN,...,NaN,0.00,N,N,N,1297-769,XXX-W549258,46500.0,2025-03-04,2025-02-26
4,26,26,3341500047,INGRAM VIVIAN,NaN,3-5,101 - RESID-SFR,1474,FORTUNE,LN,...,NaN,0.42,Y,N,N,G347-001,AN-56,5.0,2000-05-10,2000-05-09


In [14]:
df_over_1 = df[df["acreage"] > 1.0]
len(df_over_1)

1617

In [16]:
df["class_code"].value_counts().sort_index()

class_code
101 - RESID-SFR                   16753
110 - RESID-MBH                      81
120 - RESID-TWH                    1171
121 - GROUP-LIV                       5
130 - RESID-DUP/TRI                 342
160 - RESID-CNU                    3089
165 - CONDO COMMON                  172
167 - CONDO COMMON COMM              25
170 - RESID-ROW                      57
195 - COMM-APP-RES                    7
200 - SPCLTY-APT                     17
210 - SPCLTY-SMA                    107
220 - SPCLTY-TAMSBERG                29
225 - SPCLTY-CNU-TMSBRG              32
250 - SPCLTY-COMMCONDO              106
300 - BUILDNG-ONLY                    3
304 - MFG/INDUST                      2
411 - RAILRD/TRAIN                    7
451 - ROAD-ROW                       14
460 - AUTO-PARKING                   93
471 - TELEPH-COMM                     2
481 -  PUBLIC-UTIL                    5
500 - General Commercial            382
530 - SPCLTY-RTL                     19
580 - SPCLTY-RST             

| Class Code          | Count | Meaning                             |
| ------------------- | ----- | ----------------------------------- |
| 905 - VAC-RES-LOT   | 1,770 | Vacant residential lot              |
| 952 - VAC-COMM-LOT  | 121   | Vacant commercial lot               |
| 900 - RES-DEV-ACRS  | 32    | Residential development acreage     |
| 910 - COM-DEV-ACRS  | 1     | Commercial development acreage      |
| 990 - UNDEVELOPABLE | 633   | Marsh, wetlands, ROW remnants, etc. |


In [17]:
vacant = df[
    df["class_code"].isin([
        "905 - VAC-RES-LOT",
        "952 - VAC-COMM-LOT",
        "900 - RES-DEV-ACRS",
        "910 - COM-DEV-ACRS",
    ])
]

In [18]:
vacant_over_1 = vacant[vacant["acreage"] > 1.0]

len(vacant_over_1)

410

In [19]:
vacant_over_1

,id,objectid,pid,owner1,owner2,tax_district,class_code,mail_st_no,mail_st_name,mail_st_type,...,subdivision,acreage,legal_residence,other,agr,deed_book_page,plat_book_page,sale_price,recorded_date,doc_date
15,97,97,3180000055,SCOTT FRANCIS KEY Jr,NaN,5-1,905 - VAC-RES-LOT,908,SE 24TH,AVE,...,NaN,2.30,N,N,N,1015-148,XXX-UNREC S,0.0,2021-07-19,2020-05-01
28,199,199,3400700065,RILEY MARY,NaN,3-5,905 - VAC-RES-LOT,12719,QUARTERHORSE,DR,...,NaN,1.99,N,N,N,XXX-XXX,EK-709,0.0,1900-01-01,1900-01-01
146,1084,1084,3370000139,CITY OF CHARLESTON THE,NaN,3-4,905 - VAC-RES-LOT,NaN,NaN,NaN,...,NaN,13.61,N,N,N,K286-297,EB-699,5.0,1997-07-02,1997-06-30
269,2076,2076,3340500090,BATTERY ISLAND COMMUNITY LLC,NaN,3-5,905 - VAC-RES-LOT,382,RACE,ST,...,CITY OF CHARLESTON,1.04,N,N,N,-,P25- 0423,0.0,1900-01-01,1900-01-01
333,2531,2531,4270000076,VAUGHTERS JOANNE,NaN,3-1,905 - VAC-RES-LOT,1640,COOPER JUDGE LANE,NaN,...,NaN,1.27,N,N,N,M273-424,EB-271,9.0,1996-03-07,1996-02-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25234,192986,192986,4500000020,LOWCOUNTRY OPEN LAND TRUST,NaN,3-5,905 - VAC-RES-LOT,635,RUTLEDGE,AVE,...,NaN,2.00,N,N,N,Z560-333,XXX-NONE,5.0,2005-11-04,2005-10-13
25271,193263,193263,4520600126,DE GRAAF GLENN LIVING TRUST,DE GRAAF MELISSA H LIVING TRUST,3-5,905 - VAC-RES-LOT,62,CONIFER,CIR,...,NaN,1.04,N,N,N,1289-320,EH-603,975000.0,2025-01-13,2025-01-06
25623,196019,196019,3180000059,DOWD ROBERT S,DOWD TAMARA B,5-1,905 - VAC-RES-LOT,4213,SAW GRASS DRIVE,NaN,...,DIVISION 3 BLACKLOCK PLANTATIO,6.13,N,N,N,1319-984,L25- 0226,255000.0,2025-06-16,2025-06-13
25699,196615,196615,3340000103,PRIOLEAU JAMES HEIRS OF,NaN,3-1,905 - VAC-RES-LOT,31319,NaN,NaN,...,NaN,4.28,N,N,N,XXX-XXX,BO-58,0.0,1900-01-01,1900-01-01


In [22]:
vacant_over_1["subdivision"].value_counts().sort_index()

subdivision
BAYVIEW FARMS                        2
BLACKLOCK PLANTATION                 1
CENTERVILLE                          1
CENTRAL PARK CLUSTER DEVELOPMENT     1
CHARLESTO                            1
CHARLESTON                          10
CHARLESTON COUNTY                    1
CITY OF CHARLESTO                    1
CITY OF CHARLESTON                  12
CROSS CREEK TOWNHOMES                1
DIVISION 3 BLACKLOCK PLANTATIO       1
EXCHANGE PLANTATION                  2
FOLLY BEACH                          4
FOLLY WEST ESTATES                   1
HARBORWALK                           1
JAMES ISLAN                          9
JAMES ISLAND                        25
JENKINS FARM                         3
JOHNS ISLAN                          3
JOHNS ISLAND                        13
LAWTON BLUFF                         1
LIGHTHOUSE POINT                     1
MCINTYRE                             1
MILLER HILL                          1
MORRIS ISLAN                         1
OSWALD       

In [25]:
vacant_over_1[vacant_over_1["subdivision"] == "CITY OF CHARLESTON"]

,id,objectid,pid,owner1,owner2,tax_district,class_code,mail_st_no,mail_st_name,mail_st_type,...,subdivision,acreage,legal_residence,other,agr,deed_book_page,plat_book_page,sale_price,recorded_date,doc_date
269,2076,2076,3340500090,BATTERY ISLAND COMMUNITY LLC,NaN,3-5,905 - VAC-RES-LOT,382,RACE,ST,...,CITY OF CHARLESTON,1.04,N,N,N,-,P25- 0423,0.0,1900-01-01,1900-01-01
4728,36098,36098,4540700122,LENNAR CAROLINAS LLC,NaN,3-5,905 - VAC-RES-LOT,1505,KING ST EXT,NaN,...,CITY OF CHARLESTON,1.05,N,N,N,1341-696,L24- 0407,1960000.0,2025-10-02,2025-09-23
5517,41751,41751,3170000241,RUZHANSKY LLC,NaN,5-2,905 - VAC-RES-LOT,362,ELLIOT,PLACE,...,CITY OF CHARLESTON,2.97,N,N,N,1000-601,S13- 0002,259000.0,2021-06-09,2021-06-07
5999,45207,45207,3150000554,STJF LLC,NaN,5-2,905 - VAC-RES-LOT,2009,TERRABROCK,LN,...,CITY OF CHARLESTON,1.20,N,N,N,-,L23- 0308,0.0,1900-01-01,1900-01-01
7454,56330,56330,3180000555,GRATZICK BENJAMIN ELLIOT,NaN,5-2,905 - VAC-RES-LOT,2485,OAK ROYAL,DR,...,CITY OF CHARLESTON,1.50,N,N,N,1280-791,L24- 0398,5.0,2024-11-22,2024-11-14
7615,57464,57464,3450000213,SOUTH CAROLINA DEPARTMENT OF TRANSPORTATION,NaN,5-2,910 - COM-DEV-ACRS,191,NaN,NaN,...,CITY OF CHARLESTON,11.24,N,N,N,-,S16- 0234-235,0.0,1900-01-01,1900-01-01
8461,63898,63898,3430700055,CITY OF CHARLESTON,NaN,3-5,905 - VAC-RES-LOT,304,NaN,NaN,...,CITY OF CHARLESTON,3.65,N,N,N,1055-472,L24- 0221,425000.0,2021-11-24,2021-11-22
11906,90519,90519,3450000115,SOUTH CAROLINA DEPARTMENT OF TRANSPORTATION,NaN,5-2,900 - RES-DEV-ACRS,191,NaN,NaN,...,CITY OF CHARLESTON,32.40,N,N,N,0546-379,S16- 0234-235,7075000.0,2016-04-11,2016-03-23
15602,119823,119823,3450000214,SOUTH CAROLINA DEPARTMENT OF TRANSPORTATION,NaN,5-2,900 - RES-DEV-ACRS,191,NaN,NaN,...,CITY OF CHARLESTON,29.69,N,N,N,-,S16- 0234-235,0.0,1900-01-01,1900-01-01
16023,123035,123035,3460000559,STONO RIVER INVESTORS LLC,NaN,5-2,952 - VAC-COMM-LOT,49,IMMIGRATION,ST,...,CITY OF CHARLESTON,2.46,N,N,N,1059-698,L24- 0026,2392500.0,2021-12-10,2021-12-09


In [27]:
vacant_over_1.to_excel('vacant_lots_over_1_acre_within_5_miles.xlsx')